# Hands-On with Diffusion Models — Part 1

**IndabaX Uganda 2026 | Denoising Diffusion Probabilistic Models (DDPM)**

In this session, you will learn:

1. **DDPM Basics** - How diffusion models work
2. **Noise in Sampling** - Why adding noise during sampling matters
3. **Conditional Generation** - Controlling what the model generates
4. **Fast Sampling (DDIM)** - Generating images 20x faster

We'll be training on CelebA-HQ faces (1024×1024) with gender labels.

---

## Part 0: Setup

First, let's import our dependencies and load the configuration.

In [ ]:
# Standard imports
from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from IPython.display import HTML

# Our custom modules
from dataset import CelebAHQDataset, get_transform
from model import DDPMUnet
from utils import *

# Setup device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configuration
config = load_config('config.yaml')
print(f"Image size: {config['image_size']}x{config['image_size']}")
print(f"Timesteps: {config['timesteps']}")


### Load Dataset

We use CelebA-HQ with two classes: female and male.

In [ ]:
# Load training dataset (unconditional - ignoring labels for now)
train_dataset = CelebAHQDataset(
    root_dir=config['data_path'], 
    mode='train', 
    transform=get_transform(config['image_size']), 
    null_context=True  # Ignore labels for unconditional training
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=config['batch_size'], 
    shuffle=True, 
    num_workers=config['num_workers'],
    pin_memory=True
)

# Load validation dataset
val_dataset = CelebAHQDataset(
    root_dir=config['data_path'],
    mode='val',
    transform=get_transform(config['image_size']),
    null_context=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True
)

---

## Part 1: Understanding DDPM

### The Core Idea

Diffusion models work in two phases:
1. **Forward Process (Training)**: Gradually corrupt real images with Gaussian noise
2. **Reverse Process (Sampling)**: A neural network learns to remove the noise step-by-step

---

### The Forward Process

Formally, the forward process defines a conditional distribution:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1 - \bar{\alpha}_t)\mathbf{I}\right)$$

Read as: *"Given a clean image $x_0$, the noisy image at step $t$ is Gaussian — its mean is a scaled-down version of $x_0$, and variance fills in as that mean shrinks."*

**Key variables:**

| Symbol | Definition | Intuition |
|--------|-----------|----------|
| $\beta_t$ | Noise schedule | How much noise to add at step $t$. Linear: $10^{-4} \to 0.02$ over $T=500$ steps. |
| $\alpha_t = 1 - \beta_t$ | Per-step signal retention | Close to 1 (e.g. 0.98) — most signal survives each individual step. |
| $\bar{\alpha}_t = \prod_{s=1}^{t}\alpha_s$ | Cumulative signal retention | How much of $x_0$ survives after **all** $t$ steps. Starts at 1, decays to ~0 as $t \to T$. |

**From distribution to code — the reparameterisation trick:**

Any Gaussian $\mathcal{N}(\mu, \sigma^2)$ can be sampled as $\mu + \sigma\cdot\epsilon$ where $\epsilon \sim \mathcal{N}(0,I)$. Applied here:

$$\boxed{x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon}$$

This is *exactly* the distribution above, written as a computation. The key advantage: given any $x_0$ and any $t$, you can produce $x_t$ in **one step** — no need to iterate through $t$ sequential noise additions. This is what `perturb_input` implements below.

### Building the Noise Schedule

In [ ]:
# Construct DDPM noise schedule
# Beta: variance schedule (linear from beta1 to beta2)
b_t = (config['beta2'] - config['beta1']) * torch.linspace(0, 1, config['timesteps'] + 1, device=device) + config['beta1']

# Alpha: 1 - beta_t
a_t = 1 - b_t

# Alpha bar: cumulative product of alphas (computed in log space for numerical stability)
ab_t = torch.cumsum(a_t.log(), dim=0).exp()    
ab_t[0] = 1

print(f"Noise schedule created: {len(ab_t)} timesteps")
print(f"Alpha_bar at t=0: {ab_t[0]:.4f} (no noise)")
print(f"Alpha_bar at t=T/2: {ab_t[config['timesteps']//2]:.4f}")
print(f"Alpha_bar at t=T: {ab_t[-1]:.4f} (pure noise)")

### Helper Function: Adding Noise

`perturb_input` is a direct implementation of the forward equation. Each term maps exactly to the math:

$$x_t = \underbrace{\sqrt{\bar{\alpha}_t}}_{\texttt{ab\_t.sqrt()[t]}} \cdot x_0 \;+\; \underbrace{\sqrt{1-\bar{\alpha}_t}}_{\texttt{(1-ab\_t[t]).sqrt()}} \cdot \epsilon$$

In [ ]:
def perturb_input(x, t, noise):
    """
    Add noise to input x at timestep t
    Args:
        x: clean image
        t: timestep
        noise: random noise to add
    Returns:
        x_t: noisy image at timestep t
    """
    return ab_t.sqrt()[t, None, None, None] * x + (1 - ab_t[t, None, None, None]).sqrt() * noise

### Visualizing the Forward (Noising) Process

Before training a neural network to reverse the process, let's see what the **forward process** looks like — starting from real images and gradually adding noise until they become pure Gaussian noise.

Each frame steps forward through the timesteps $t = 0 \to T$, showing how the signal decays into noise.

In [ ]:
# Load 4 real images from the dataset
real_images, _ = next(iter(DataLoader(train_dataset, batch_size=4, shuffle=True)))
real_images = real_images.to(device)

# Fix the noise so every frame uses the same noise vector — only the scale changes
noise = torch.randn_like(real_images)

# Record the image at every save_rate timesteps (t=0 is the clean image, t=T is pure noise)
save_rate = 20
noise_intermediate = []

for t in range(0, config['timesteps'] + 1, save_rate):
    x_t = perturb_input(real_images, t, noise)
    noise_intermediate.append(x_t.detach().cpu().numpy())

noise_intermediate = np.stack(noise_intermediate)  # (n_frames, 4, 3, H, W)

# Animate — watch real faces gradually dissolve into noise
plt.clf()
animation_forward = plot_sample(noise_intermediate, 4, 2, config['save_dir'], "forward_noise", None, save=False)
HTML(animation_forward.to_jshtml())

### The Neural Network

We use a U-Net architecture that:
- Takes a noisy image $x_t$ and timestep $t$ as input
- Predicts the noise $\epsilon$ that was added
- Optionally takes context $c$ (e.g., "male" or "female") for conditional generation

In [ ]:
# Initialize the model
nn_model = DDPMUnet(
    in_channels=3,  # RGB images
    n_feat=config['n_feat'],  # Feature dimensions
    n_cfeat=config['n_cfeat'],  # Context dimensions (2 for male/female)
    image_size=config['image_size']  # Image resolution
).to(device)

print(f"Model created with {sum(p.numel() for p in nn_model.parameters()):,} parameters")

### Training Unconditional DDPM

Now let's implement the full training pipeline with validation and model checkpointing.

**Training Loop Logic:**
1. Sample a random timestep $t$ for each image
2. Add noise: $x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon$
3. Model predicts the noise: $\hat{\epsilon} = f_\theta(x_t, t)$
4. Minimise the loss: $\mathcal{L} = \|\epsilon - \hat{\epsilon}\|^2$
5. Validate and save best model

**Why is MSE on noise the right loss?**  
The DDPM objective is derived from the ELBO (evidence lower bound). With the noise parameterisation, all the KL divergence terms in the ELBO reduce to a single, simple expression:

$$\mathcal{L}_\text{simple} = \mathbb{E}_{t,\, x_0,\, \epsilon}\!\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

The deep variational derivation collapses to plain MSE between the noise we added and the noise the network predicted. Note: **the reverse process is never evaluated during training** — we only use the forward equation to create noisy images.

In [ ]:
def train_unconditional_ddpm(model, train_loader, val_loader, config, device):
    optim = torch.optim.Adam(model.parameters(), lr=config['lrate'])
    os.makedirs(config['save_dir'], exist_ok=True)

    start_ep, best_val_loss = load_checkpoint(config['save_dir'], 'model', model, optim, device)
    if start_ep >= config['n_epoch']:
        print('Training already completed.'); return

    for ep in range(start_ep, config['n_epoch']):
        print(f"\n{'='*60}\nEpoch {ep+1}/{config['n_epoch']}\n{'='*60}")

        # Linearly decay learning rate
        optim.param_groups[0]['lr'] = config['lrate'] * (1 - ep / config['n_epoch'])

        # === TRAINING ===
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc='Training', leave=False)
        for x, _ in pbar:
            optim.zero_grad()
            x = x.to(device)
            noise = torch.randn_like(x)
            t = torch.randint(1, config['timesteps'] + 1, (x.shape[0],), device=device)
            loss = F.mse_loss(model(perturb_input(x, t, noise), t / config['timesteps']), noise)
            loss.backward(); optim.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f"Train Loss: {train_loss / len(train_loader):.4f}")

        # === VALIDATION ===
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            pbar = tqdm(val_loader, desc='Validation', leave=False)
            for x, _ in pbar:
                x = x.to(device)
                noise = torch.randn_like(x)
                t = torch.randint(1, config['timesteps'] + 1, (x.shape[0],), device=device)
                val_loss += F.mse_loss(model(perturb_input(x, t, noise), t / config['timesteps']), noise).item()
                pbar.set_postfix({'loss': f'{val_loss:.4f}'})
        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation Loss: {avg_val_loss:.4f}")

        # === SAVE CHECKPOINTS ===
        if (ep + 1) % 4 == 0 or ep == config['n_epoch'] - 1:
            save_checkpoint(config['save_dir'], 'model', ep, model, optim, best_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(config['save_dir'], 'model_best.pth'))
            print(f"\u2713 New best model! Val Loss: {best_val_loss:.4f}")

    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

In [ ]:
SKIP_TRAINING = True   # ← Set to False to train

if not SKIP_TRAINING:
    train_unconditional_ddpm(nn_model, train_loader, val_loader, config, device)


---

## Part 2: Sampling with DDPM

### The Reverse Process

The network learns to run diffusion backwards. Formally:

$$p_\theta(\mathbf{x}_{t-1} \mid \mathbf{x}_t) = \mathcal{N}\!\left(\mathbf{x}_{t-1};\; \boldsymbol{\mu}_\theta(\mathbf{x}_t, t),\; \sigma_t^2\,\mathbf{I}\right)$$

Same Gaussian form as the forward process, but running backward in time and with the mean $\boldsymbol{\mu}_\theta$ **predicted by the U-Net**. The $\theta$ subscript means the network weights are the learnable parameters.

**Why predict noise $\hat{\epsilon}$ rather than the mean directly?**  
Ho et al. (2020) showed this works better. The predicted noise converts to the mean via Bayes applied to Gaussians:

$$\boldsymbol{\mu}_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\,\hat{\epsilon}\right)$$

> $\alpha_t = 1-\beta_t$ is the **per-step** alpha; $\bar{\alpha}_t$ is **cumulative** — they play different roles here.

### The Sampling Algorithm

Applying the reparameterisation trick to $p_\theta$ gives the practical update rule:

$$\boxed{x_{t-1} = \underbrace{\frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\hat{\epsilon}\right)}_{\text{denoised estimate } (\boldsymbol{\mu}_\theta)} + \underbrace{\sigma_t z}_{\text{stochastic injection}}}$$

where $z \sim \mathcal{N}(0,I)$ and $\hat{\epsilon} = f_\theta(x_t, t)$.

**Each term:**
- **Denoised estimate**: take $x_t$, subtract the predicted noise (scaled appropriately), undo the per-step signal scaling
- **$\sigma_t z$**: fresh noise injected every step — prevents all samples from collapsing to a single deterministic path. This is the term **DDIM removes** (Part 5) to get fast, deterministic sampling.

**Full algorithm:**
1. Sample $x_T \sim \mathcal{N}(0, I)$
2. For $t = T, T-1, \ldots, 1$: apply the reverse step above
3. Return $x_0$

In [ ]:
def denoise_add_noise(x, t, pred_noise, z=None):
    """
    Remove predicted noise from x and add a small amount of random noise (except at t=1)
    
    Args:
        x: current noisy image
        t: current timestep
        pred_noise: predicted noise from model
        z: random noise (if None, sample new noise)
    Returns:
        x_{t-1}: less noisy image
    """
    if z is None:
        z = torch.randn_like(x)
    
    noise = b_t.sqrt()[t] * z  # σ_t * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise

In [ ]:
@torch.no_grad()
def sample_ddpm(n_sample, save_rate=20):
    """
    Generate images using DDPM sampling
    
    Args:
        n_sample: number of images to generate
        save_rate: save intermediate steps every save_rate timesteps
    Returns:
        samples: final generated images
        intermediate: list of intermediate denoising steps
    """
    # Start from pure noise
    samples = torch.randn(n_sample, 3, config['image_size'], config['image_size']).to(device)
    
    intermediate = []
    
    # Denoise step by step
    for i in range(config['timesteps'], 0, -1):
        print(f'Sampling timestep {i:3d}', end='\r')
        
        t = torch.tensor([i / config['timesteps']])[:, None, None, None].to(device)
        
        # Don't add noise at the last step (t=1)
        z = torch.randn_like(samples) if i > 1 else 0
        
        # Predict and remove noise
        eps = nn_model(samples, t)
        samples = denoise_add_noise(samples, i, eps, z)
        
        # Save intermediate steps for visualization
        if i % save_rate == 0 or i == config['timesteps'] or i < 8:
            intermediate.append(samples.detach().cpu().numpy())
    
    intermediate = np.stack(intermediate)
    return samples, intermediate

### Load Pre-trained Model and Sample

In [ ]:
# Load best trained weights
model_path = os.path.join(config['save_dir'], "model_best.pth")
# Or use specific epoch: model_path = os.path.join(config['save_dir'], "model_31.pth")

nn_model.load_state_dict(torch.load(model_path, map_location=device))
nn_model.eval()
print(f"Loaded pre-trained model from {model_path}")

In [ ]:
# Generate samples
plt.clf()
samples, intermediate = sample_ddpm(4)
animation = plot_sample(intermediate, 4, 2, config['save_dir'], "ddpm_basic", None, save=False)
HTML(animation.to_jshtml())

---

## Part 3: The Role of Noise in Sampling

### Experiment: What happens without extra noise?

Let's compare sampling **with** and **without** the extra stochastic noise term.

In [ ]:
@torch.no_grad()
def sample_ddpm_no_noise(n_sample, save_rate=20):
    """
    INCORRECT sampling: doesn't add noise back in (deterministic)
    This will show the importance of stochasticity!
    """
    samples = torch.randn(n_sample, 3, config['image_size'], config['image_size']).to(device)
    intermediate = []
    
    for i in range(config['timesteps'], 0, -1):
        print(f'Sampling timestep {i:3d}', end='\r')
        
        t = torch.tensor([i / config['timesteps']])[:, None, None, None].to(device)
        
        # DIFFERENCE: Always use z=0 (no added noise)
        z = 0
        
        eps = nn_model(samples, t)
        samples = denoise_add_noise(samples, i, eps, z)
        
        if i % save_rate == 0 or i == config['timesteps'] or i < 8:
            intermediate.append(samples.detach().cpu().numpy())
    
    intermediate = np.stack(intermediate)
    return samples, intermediate

### Compare: With vs Without Noise

In [ ]:
# Sample WITHOUT extra noise (deterministic)
plt.clf()
samples_no_noise, intermediate_no_noise = sample_ddpm_no_noise(4)
animation_no_noise = plot_sample(intermediate_no_noise, 4, 2, config['save_dir'], "ddpm_no_noise", None, save=False)
print("\n--- Without Extra Noise (Deterministic) ---")
HTML(animation_no_noise.to_jshtml())

In [ ]:
# Sample WITH extra noise (stochastic - correct way)
plt.clf()
samples_with_noise, intermediate_with_noise = sample_ddpm(4)
animation_with_noise = plot_sample(intermediate_with_noise, 4, 2, config['save_dir'], "ddpm_with_noise", None, save=False)
print("\n--- With Extra Noise (Stochastic - Correct) ---")
HTML(animation_with_noise.to_jshtml())

### Observations

**Without noise:** Images may appear blurry or mode-collapsed.

**With noise:** Better diversity and quality!

The stochastic noise ensures the sampling process explores multiple pathways instead of converging to a single deterministic trajectory.

---

## Part 4: Conditional Generation with Context

### The Problem

So far, we can generate faces, but we can't control **what kind** of face (male/female).

### The Solution: Context Conditioning

We add a context vector $c$ to the model:
- Female: $c = [1, 0]$
- Male: $c = [0, 1]$

The model becomes: $\epsilon = f(x_t, t, c)$

### Classifier-Free Guidance

During training, we **randomly** set $c = 0$ (90% of the time). This allows:
1. The model to learn unconditional generation
2. Better control during sampling via guidance

Let's train a context-aware model:

In [ ]:
# Load dataset WITH labels for conditional training
train_dataset_context = CelebAHQDataset(
    root_dir=config['data_path'],
    mode='train',
    transform=get_transform(config['image_size']),
    null_context=False  # Use actual gender labels
)

train_loader_context = DataLoader(
    train_dataset_context,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    pin_memory=True
)

val_dataset_context = CelebAHQDataset(
    root_dir=config['data_path'],
    mode='val',
    transform=get_transform(config['image_size']),
    null_context=False
)

val_loader_context = DataLoader(
    val_dataset_context,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True
)

### Training Conditional DDPM with Context

In [ ]:
def train_conditional_ddpm(model, train_loader, val_loader, config, device, context_mask_prob=0.1):
    optim = torch.optim.Adam(model.parameters(), lr=config['lrate'])
    os.makedirs(config['save_dir'], exist_ok=True)

    start_ep, best_val_loss = load_checkpoint(config['save_dir'], 'context_model', model, optim, device)
    if start_ep >= config['n_epoch']:
        print('Training already completed.'); return

    for ep in range(start_ep, config['n_epoch']):
        print(f"\n{'='*60}\nEpoch {ep+1}/{config['n_epoch']}\n{'='*60}")

        # Linearly decay learning rate
        optim.param_groups[0]['lr'] = config['lrate'] * (1 - ep / config['n_epoch'])

        # === TRAINING ===
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc='Training', leave=False)
        for x, c in pbar:
            optim.zero_grad()
            x, c = x.to(device), c.to(device).float()
            # Classifier-free guidance: randomly drop context with probability context_mask_prob
            c = c * torch.bernoulli(torch.full((c.shape[0],), 1 - context_mask_prob, device=device)).unsqueeze(-1)
            noise = torch.randn_like(x)
            t = torch.randint(1, config['timesteps'] + 1, (x.shape[0],), device=device)
            loss = F.mse_loss(model(perturb_input(x, t, noise), t / config['timesteps'], c), noise)
            loss.backward(); optim.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f"Train Loss: {train_loss / len(train_loader):.4f}")

        # === VALIDATION ===
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            pbar = tqdm(val_loader, desc='Validation', leave=False)
            for x, c in pbar:
                x, c = x.to(device), c.to(device).float()
                noise = torch.randn_like(x)
                t = torch.randint(1, config['timesteps'] + 1, (x.shape[0],), device=device)
                val_loss += F.mse_loss(model(perturb_input(x, t, noise), t / config['timesteps'], c), noise).item()
                pbar.set_postfix({'loss': f'{val_loss:.4f}'})
        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation Loss: {avg_val_loss:.4f}")

        # === SAVE CHECKPOINTS ===
        if (ep + 1) % 4 == 0 or ep == config['n_epoch'] - 1:
            save_checkpoint(config['save_dir'], 'context_model', ep, model, optim, best_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(config['save_dir'], 'context_model_best.pth'))
            print(f"\u2713 New best model! Val Loss: {best_val_loss:.4f}")

    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

In [ ]:
SKIP_TRAINING_CONTEXT = True   # ← Set to False to train

if not SKIP_TRAINING_CONTEXT:
    nn_model_context = DDPMUnet(
        in_channels=3,
        n_feat=config['n_feat'],
        n_cfeat=config['n_cfeat'],
        image_size=config['image_size']
    ).to(device)
    train_conditional_ddpm(nn_model_context, train_loader, val_loader, config, device)


### Sampling with Context

In [ ]:
@torch.no_grad()
def sample_ddpm_context(n_sample, context, save_rate=20):
    """
    Sample with context conditioning
    
    Args:
        n_sample: number of samples
        context: context tensor (n_sample, 2) or (2,) 
    """
    samples = torch.randn(n_sample, 3, config['image_size'], config['image_size']).to(device)
    
    # Prepare context
    if len(context.shape) == 1:
        context = context.unsqueeze(0).repeat(n_sample, 1)
    c = context.to(device)
    
    intermediate = []
    
    for i in range(config['timesteps'], 0, -1):
        print(f'Sampling timestep {i:3d}', end='\r')
        
        t = torch.tensor([i / config['timesteps']])[:, None, None, None].to(device)
        z = torch.randn_like(samples) if i > 1 else 0
        
        # Predict noise WITH context
        eps = nn_model(samples, t, c)
        samples = denoise_add_noise(samples, i, eps, z)
        
        if i % save_rate == 0 or i == config['timesteps'] or i < 8:
            intermediate.append(samples.detach().cpu().numpy())
    
    intermediate = np.stack(intermediate)
    return samples, intermediate

In [ ]:
# Load context-aware model (best model)
context_model_path = os.path.join(config['save_dir'], "context_model_best.pth")
# Or use specific epoch: context_model_path = os.path.join(config['save_dir'], "context_model_31.pth")

nn_model.load_state_dict(torch.load(context_model_path, map_location=device))
nn_model.eval()
print(f"Loaded context-aware model from {context_model_path}")

In [ ]:
# Define context vectors
ctx_female = torch.tensor([1, 0], dtype=torch.float32)
ctx_male = torch.tensor([0, 1], dtype=torch.float32)

print("Context vectors:")
print(f"Female: {ctx_female.tolist()}")
print(f"Male: {ctx_male.tolist()}")

### Generate Female Faces

In [ ]:
plt.clf()
samples_female, intermediate_female = sample_ddpm_context(4, ctx_female)
animation_female = plot_sample(intermediate_female, 4, 2, config['save_dir'], "female", None, save=False)
print("\n--- Female Faces ---")
HTML(animation_female.to_jshtml())

### Generate Male Faces

In [ ]:
plt.clf()
samples_male, intermediate_male = sample_ddpm_context(4, ctx_male)
animation_male = plot_sample(intermediate_male, 4, 2, config['save_dir'], "male", None, save=False)
print("\n--- Male Faces ---")
HTML(animation_male.to_jshtml())

### Mix Contexts

We can even blend contexts! Try $c = [0.5, 0.5]$ or $c = [0.7, 0.3]$

In [ ]:
# Helper function for showing images
def show_images(imgs, nrow=4):
    """
    Display a grid of images
    """
    n_imgs = imgs.shape[0]
    ncols = min(nrow, n_imgs)
    nrows = (n_imgs + ncols - 1) // ncols
    
    _, axs = plt.subplots(ncols, nrows, figsize=(nrows * 4, ncols * 4))
    
    if n_imgs == 1:
        axs = [axs]
    else:
        axs = axs.flatten()
    
    for i, (img, ax) in enumerate(zip(imgs, axs)):
        img_np = (img.permute(1, 2, 0).clip(-1, 1).detach().cpu().numpy() + 1) / 2
        ax.imshow(img_np)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"Image {i+1}")
    
    # Hide unused subplots
    for ax in axs[n_imgs:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Custom mixed contexts
ctx_mixed = torch.tensor([
    [1.0, 0.0],  # Pure female
    [0.8, 0.2],  # Mostly female
    [0.6, 0.4],
    [0.5, 0.5],  # Balanced
    [0.4, 0.6],
    [0.2, 0.8],  # Mostly male
    [0.0, 1.0],  # Pure male
    [0.0, 0.0],  # Unconditional
], dtype=torch.float32)

samples_mixed, _ = sample_ddpm_context(8, ctx_mixed)
show_images(samples_mixed, nrow=2)

---

## Part 5: Fast Sampling with DDIM

### The Problem with DDPM

DDPM requires $T=500$ (or more) steps to generate one image. This is **slow**!

### The Solution: DDIM (Denoising Diffusion Implicit Models)

DDIM allows us to skip timesteps! Instead of going $T \rightarrow T-1 \rightarrow ... \rightarrow 1$, we can jump: $T \rightarrow T-25 \rightarrow T-50 \rightarrow ... \rightarrow 1$

**Key Insight:** DDIM makes the sampling process **deterministic** (no added noise), which allows larger steps.

### DDIM Update Rule

Instead of the stochastic update, DDIM uses:
$$x_{t-\Delta t} = \sqrt{\bar{\alpha}_{t-\Delta t}} \cdot \frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} + \sqrt{1-\bar{\alpha}_{t-\Delta t}}\epsilon_\theta(x_t, t)$$

This is deterministic - no random noise added!

In [ ]:
def denoise_ddim(x, t, t_prev, pred_noise):
    """
    DDIM denoising step (deterministic)
    
    Args:
        x: current noisy image
        t: current timestep
        t_prev: previous timestep (can be > 1 step back!)
        pred_noise: predicted noise
    Returns:
        x_prev: denoised image at t_prev
    """
    ab = ab_t[t]
    ab_prev = ab_t[t_prev]
    
    # Predict x0 from xt
    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
    
    # Direction from x0 to xt_prev
    dir_xt = (1 - ab_prev).sqrt() * pred_noise
    
    # Combine
    return ab_prev.sqrt() * x0_pred + dir_xt

In [ ]:
@torch.no_grad()
def sample_ddim(n_sample, n_steps=25, context=None):
    """
    Fast sampling with DDIM
    
    Args:
        n_sample: number of samples
        n_steps: number of denoising steps (much less than timesteps!)
        context: optional context vector
    """
    samples = torch.randn(n_sample, 3, config['image_size'], config['image_size']).to(device)
    
    # Prepare context
    c = None
    if context is not None:
        if len(context.shape) == 1:
            context = context.unsqueeze(0).repeat(n_sample, 1)
        c = context.to(device)
    
    intermediate = []
    step_size = config['timesteps'] // n_steps
    
    for i in range(config['timesteps'], 0, -step_size):
        print(f'Sampling timestep {i:3d}', end='\r')
        
        t = torch.tensor([i / config['timesteps']])[:, None, None, None].to(device)
        
        # Predict noise
        eps = nn_model(samples, t, c)
        
        # DDIM step (deterministic, can skip timesteps)
        samples = denoise_ddim(samples, i, max(0, i - step_size), eps)
        
        intermediate.append(samples.detach().cpu().numpy())
    
    intermediate = np.stack(intermediate)
    return samples, intermediate

### Compare Speed: DDPM vs DDIM

In [ ]:
import time

# Benchmark DDPM (500 steps)
start = time.time()
_, _ = sample_ddpm(4, save_rate=100)
ddpm_time = time.time() - start

# Benchmark DDIM (25 steps)
start = time.time()
_, _ = sample_ddim(4, n_steps=25)
ddim_time = time.time() - start

print(f"\nDDPM (500 steps): {ddpm_time:.2f}s")
print(f"DDIM (25 steps): {ddim_time:.2f}s")
print(f"Speedup: {ddpm_time/ddim_time:.1f}x faster!")

### Visualize DDIM Sampling

In [ ]:
plt.clf()
samples_ddim, intermediate_ddim = sample_ddim(4, n_steps=25)
animation_ddim = plot_sample(intermediate_ddim, 4, 2, config['save_dir'], "ddim", None, save=False)
print("\n--- DDIM Sampling (25 steps) ---")
HTML(animation_ddim.to_jshtml())

### DDIM with Context

In [ ]:
# Fast female faces
plt.clf()
samples_ddim_female, intermediate_ddim_female = sample_ddim(4, n_steps=25, context=ctx_female)
animation_ddim_female = plot_sample(intermediate_ddim_female, 4, 2, config['save_dir'], "ddim_female", None, save=False)
print("\n--- DDIM Female Faces (25 steps) ---")
HTML(animation_ddim_female.to_jshtml())

In [ ]:
# Fast male faces
plt.clf()
samples_ddim_male, intermediate_ddim_male = sample_ddim(4, n_steps=25, context=ctx_male)
animation_ddim_male = plot_sample(intermediate_ddim_male, 4, 2, config['save_dir'], "ddim_male", None, save=False)
print("\n--- DDIM Male Faces (25 steps) ---")
HTML(animation_ddim_male.to_jshtml())

---

## Part 6: Summary & Interactive Playground

### What We Learned

1. **DDPM Basics**
   - Forward process: gradually add noise
   - Reverse process: learn to denoise
   - Training: predict the noise that was added

2. **Importance of Stochastic Noise**
   - Adding noise during sampling prevents collapse
   - Enables exploration of multiple generation paths

3. **Conditional Generation**
   - Context vectors control what gets generated
   - Classifier-free guidance enables controllable generation
   - Can mix contexts for interpolation

4. **Fast Sampling with DDIM**
   - Deterministic sampling allows skipping steps
   - 20x speedup (25 steps vs 500 steps)
   - Works with context conditioning

### Interactive Playground

In [ ]:
def generate_faces(n_faces=8, method='ddim', n_steps=25, female_weight=0.5):
    """
    Interactive generation function
    
    Args:
        n_faces: number of faces to generate
        method: 'ddpm' or 'ddim'
        n_steps: number of DDIM steps (if method='ddim')
        female_weight: 0.0=male, 1.0=female, 0.5=mixed
    """
    # Create context
    ctx = torch.tensor([female_weight, 1 - female_weight], dtype=torch.float32)
    
    print(f"Generating {n_faces} faces with {method.upper()}...")
    print(f"Context: female={female_weight:.1f}, male={1-female_weight:.1f}")
    
    if method == 'ddpm':
        samples, _ = sample_ddpm_context(n_faces, ctx, save_rate=100)
    else:  # ddim
        samples, _ = sample_ddim(n_faces, n_steps=n_steps, context=ctx)
    
    show_images(samples, nrow=2)

In [ ]:
# Example 1: Fast female faces
generate_faces(n_faces=8, method='ddim', n_steps=25, female_weight=1.0)

In [ ]:
# Example 2: Fast male faces
generate_faces(n_faces=8, method='ddim', n_steps=25, female_weight=0.0)

In [ ]:
# Example 3: Mixed/ambiguous faces
generate_faces(n_faces=8, method='ddim', n_steps=25, female_weight=0.5)

In [ ]:
# Example 4: Very fast sampling (10 steps)
generate_faces(n_faces=8, method='ddim', n_steps=10, female_weight=0.7)

In [ ]:
# Example 5: High quality (50 steps)
generate_faces(n_faces=8, method='ddim', n_steps=50, female_weight=0.3)

---

## Appendix: The Reparameterization Trick

Throughout this tutorial you may have noticed lines like:

```python
noise = torch.randn_like(x)          # ε ~ N(0, I)
x_t = sqrt_alpha_bar * x + sqrt_one_minus_alpha_bar * noise
```

This is the **reparameterization trick** in action — and it is the key that makes training diffusion models (and VAEs) possible.

### The Problem: Sampling Breaks Gradients

Training a neural network requires backpropagation — computing gradients through every operation.
But sampling from a distribution (e.g. drawing a noisy image) is a *stochastic* step:
you cannot differentiate through pure randomness, so gradients cannot flow back through it.

### The Solution: Outsource the Randomness

Instead of sampling $x_t$ directly from $\mathcal{N}(\mu_t,\,\sigma_t^2 I)$, we rewrite it as:

$$x_t = \mu_t + \sigma_t \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

Now $\varepsilon$ is sampled from a *fixed* standard Gaussian that has **no learnable parameters**.
Everything the model cares about — $\mu_t$ and $\sigma_t$ (or their schedule equivalents $\bar{\alpha}_t$) — is in the deterministic part.
Gradients flow freely through the deterministic transformation; the randomness is safely isolated in $\varepsilon$.

### Why It Matters for Diffusion Models

| Where | What the trick enables |
|---|---|
| **Forward process** (`perturb_input`) | Corrupt any image to $x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon$ in one shot — no loop needed, gradient-friendly |
| **Reverse process** (`denoise_add_noise`) | Controlled re-injection of noise $z$ at each denoising step to keep sampling stochastic |
| **Loss function** | We optimise $\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2$ — the model learns to *predict the noise*, which is only possible because we factored it out |

### Broader Importance

The reparameterization trick is foundational to modern generative modelling:

- **VAEs** — makes the latent sampling differentiable so the encoder and decoder can be trained jointly.
- **Diffusion models** — enables closed-form single-step corruption and gradient-based noise prediction.
- **Flow models & score matching** — same principle underlies continuous-time formulations.

Without it, the training objectives of all these models would be non-differentiable and gradient descent would simply not work.


---

## Acknowledgments

This tutorial is based on:
- **Paper**: [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) (Ho et al., 2020)
- **Paper**: [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502) (Song et al., 2020)
- **Code**: Modified from [minDiffusion](https://github.com/cloneofsimo/minDiffusion)
- **Dataset**: CelebA-HQ

### Further Reading

- Classifier-Free Guidance: [Paper](https://arxiv.org/abs/2207.12598)
- Latent Diffusion Models (Stable Diffusion): [Paper](https://arxiv.org/abs/2112.10752)
- Score-Based Models: [Paper](https://arxiv.org/abs/2011.13456)

---

**Questions? Experiments? Go ahead and explore!**